# Pipeline executing Isolation Foerest Outliers and k-NN Imputing of the missing values

In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Importing Data

In [2]:
url2 = 'https://raw.githubusercontent.com/JeroenGuillierme/Project-MDA/main/Data/'

rta_df = pd.read_csv(
    f'{url2}total_df_with_distances.csv')

interventions_data = rta_df[rta_df['Intervention'] == 1]
print('Missing values:\n', interventions_data.isna().sum())

## 2. Importing custom transformers for outlier removal and missing values imputation

**DataFrameSplitter**

* Splits dataframe in two, one containing NaNs and one without NaNs

**IsolationForestFilter**

* The custom transformer IsolationForestOutlierRemoval allows the Isolation Forest to be integrated into a scikit-learn pipeline by providing both fit and transform methods.
* It removes the outliers detected by the IsolationForest and passes the cleaned data to the next step (like KNN imputation).
* This approach ensures that you can streamline the entire preprocessing workflow, including outlier removal and imputation, in a single pipeline.

**KNNImputerByGroup**

* Splits the data per vector type.
* Imputes missing values based on selected feautures: Latitude, Longitude, distance to specified vector type and T3-T0
* Concatenates groups back together

In [3]:
from CustomTransformer import DataFrameSplitter, IsolationForestFilter, KNNImputerByGroup

## 3. Creating two Pipelines

In [4]:
# set seed for allowing multiple runs with same outcome
np.random.seed(42)

# Defining the Features for Each Group
feature_groups = {
    'Ambulance': ['Latitude', 'Longitude', 'distance_to_ambulance', 'T3-T0'],
    'MUG': ['Latitude', 'Longitude', 'distance_to_mug', 'T3-T0'],
    'PIT': ['Latitude', 'Longitude', 'distance_to_pit', 'T3-T0']
}

# Pipeline for Filtering Inliers
filtering_pipeline = Pipeline(steps=[
    ('splitter', DataFrameSplitter(column='T3-T0')),
    ('isolation_forest', IsolationForestFilter(column='T3-T0'))
])

# Pipeline for Imputation
imputation_pipeline = Pipeline(steps=[
    ('knn_imputer', KNNImputerByGroup(group_column='Vector type', feature_groups=feature_groups))
])


In [5]:
# Process Data through the Pipeline
# Split the DataFrame
without_nan, with_nan = filtering_pipeline.named_steps['splitter'].transform(interventions_data)

# Filter Inliers using Isolation Forest
filtered_data = filtering_pipeline.named_steps['isolation_forest'].fit_transform(without_nan)

# Combine with the rows that had NaN values
combined_data = pd.concat([filtered_data, with_nan], axis=0)

# Reset index
combined_data.reset_index(drop=True, inplace=True)

# Impute Missing Values
final_data = imputation_pipeline.named_steps['knn_imputer'].fit_transform(combined_data)

# Reset index
final_data.reset_index(drop=True, inplace=True)

# Checking for any remaining missing values
print('Missing values:\n', final_data.isna().sum())

In [6]:
final_data

In [7]:
# Plot response times
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(data=final_data, x='T3-T0', bins=50, log_scale=False, kde=True, hue='Vector type',ax=axes[0]).set(
    title='Imputed Response Times', xlabel='T3-T0') 
sns.histplot(data=final_data, x='T3-T0', bins=50, log_scale=True, kde=True, hue='Vector type',ax=axes[1]).set(
    title='Logscale of Imputed Response Times', xlabel='Log10(T3-T0)') 
# Right-skewed distribution, so log scale was used.